# Analyses historiques sur 5 ans : météo, pollution et leur croisement

Ce notebook exploite les **archives** transformées en séries horaires (`scripts/archives_to_gold.py`) : météo Open-Meteo 2022 à 2026 avec visibilité, pollution OpenAQ 2022 à 2026 (archive publique S3). Il répond, sur plusieurs années, aux questions métier des Parties 1, 3 et 4 qui n'étaient jusque-là traitées que sur quelques semaines.

**Sources et licences :** Open-Meteo.com (CC BY 4.0), OpenAQ (archive publique). Horodatage OpenAQ : l'archive donne la fin de la période d'une heure ; elle est ramenée au début de période (-1 h), comme le Silver en direct, après comparaison avec l'API.

**Limites à garder en tête :**
- les stations de pollution sont celles retenues par le script (5 emplacements les plus complets par ville), pas tout le réseau ;
- la météo est une **réanalyse modélisée** (maille de 9 à 25 km), pas une mesure de station ; la visibilité aussi ;
- Lille n'a de visibilité qu'à partir du 26 juin 2024 ; Paris et Lyon depuis 2022 ;
- l'archive OpenAQ s'arrête 72 heures avant la date de génération.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config" / "cities.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA = PROJECT_ROOT / "data" / "archives-gold"
FUSEAU = "Europe/Paris"
SEUILS_OMS = {"pm25": 15.0, "pm10": 45.0, "no2": 25.0, "o3": 100.0, "co": 4000.0}   # lignes directrices 24 h, µg/m³
NOMS = {c["id"]: c["name"] for c in yaml.safe_load((PROJECT_ROOT / "config" / "cities.yml").read_text(encoding="utf-8"))["cities"]}

manquants = [n for n in ("weather_daily_visibility", "pollution_hourly_archive", "pollution_monthly", "city_hourly_history")
             if not (DATA / f"{n}.parquet").exists()]
if manquants:
    raise RuntimeError(f"Tables absentes de {DATA}: {manquants}. Lancer scripts/openmeteo_archive.py, scripts/openaq_archive.py --silver puis scripts/archives_to_gold.py.")

jour = pd.read_parquet(DATA / "weather_daily_visibility.parquet")
poll = pd.read_parquet(DATA / "pollution_hourly_archive.parquet")
mensuel = pd.read_parquet(DATA / "pollution_monthly.parquet")
hist = pd.read_parquet(DATA / "city_hourly_history.parquet")
for df, col in ((jour, "day_utc"), (poll, "hour_utc"), (hist, "hour_utc"), (mensuel, "month")):
    df[col] = pd.to_datetime(df[col], utc=True)

print("Météo journalière :", f"{len(jour):,} jours-ville, du {jour.day_utc.min():%d/%m/%Y} au {jour.day_utc.max():%d/%m/%Y}")
print("Pollution horaire :", f"{len(poll):,} lignes ville-polluant-heure, du {poll.hour_utc.min():%d/%m/%Y} au {poll.hour_utc.max():%d/%m/%Y}")
print("Jointure :", hist["statut_jointure"].value_counts().to_dict())
cov = poll.assign(annee=poll.hour_utc.dt.year).groupby(["city_id", "annee"])["hour_utc"].nunique().unstack(fill_value=0)
print("\nHeures avec mesure de pollution, par ville et année :")
display(cov)

## 1. Périodes météorologiques anormales (Partie 1, question 1)

Une journée est **anormale** si sa température maximale dépasse le 95e percentile de sa ville et de sa saison (chaleur), ou si sa température minimale est sous le 5e percentile (froid). Un **épisode** est une suite d'au moins 3 jours anormaux consécutifs. Les seuils sont calculés sur 2022 à 2026.

In [ ]:
jour = jour.sort_values(["city_id", "day_utc"]).copy()
jour["saison"] = jour["day_utc"].dt.month.map(lambda m: "hiver" if m in (12, 1, 2) else "printemps" if m in (3, 4, 5) else "été" if m in (6, 7, 8) else "automne")
seuils = jour.groupby(["city_id", "saison"]).agg(p95_tmax=("temperature_max", lambda s: s.quantile(0.95)), p05_tmin=("temperature_min", lambda s: s.quantile(0.05))).reset_index()
jour = jour.merge(seuils, on=["city_id", "saison"])
jour["chaud"] = jour["temperature_max"] > jour["p95_tmax"]
jour["froid"] = jour["temperature_min"] < jour["p05_tmin"]

def episodes(df, flag, min_jours=3):
    lignes = []
    for ville, g in df.sort_values("day_utc").groupby("city_id"):
        run = (g[flag] != g[flag].shift()).cumsum()
        for _, bloc in g[g[flag]].groupby(run[g[flag]]):
            if len(bloc) >= min_jours:
                lignes.append({"ville": NOMS[ville], "type": "chaud" if flag == "chaud" else "froid", "debut": bloc.day_utc.min().date(),
                               "fin": bloc.day_utc.max().date(), "jours": len(bloc),
                               "extreme": round(bloc["temperature_max"].max() if flag == "chaud" else bloc["temperature_min"].min(), 1)})
    return pd.DataFrame(lignes)

ep = pd.concat([episodes(jour, "chaud"), episodes(jour, "froid")], ignore_index=True)
print(f"{len(ep)} épisodes d'au moins 3 jours anormaux sur 2022 à 2026")
display(ep.sort_values("jours", ascending=False).head(12).reset_index(drop=True))

## 2. Évolution saisonnière de la température (Partie 1, question 3)

In [ ]:
jour["annee"] = jour["day_utc"].dt.year
jour["mois"] = jour["day_utc"].dt.month
clim = jour[jour["annee"] < 2026].groupby(["city_id", "mois"])["temperature_mean"].mean().unstack(0)
clim.columns = [NOMS[c] for c in clim.columns]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
clim.plot(ax=axes[0], marker="o")
axes[0].set_title("Climatologie mensuelle 2022-2025 (température moyenne, °C)"); axes[0].set_xticks(range(1, 13)); axes[0].set_xlabel("Mois"); axes[0].grid(True, linestyle="--", alpha=0.4)
annuel = jour[jour["annee"] < 2026].groupby(["annee", "city_id"])["temperature_mean"].mean().unstack()
annuel.columns = [NOMS[c] for c in annuel.columns]
annuel.plot(kind="bar", ax=axes[1]); axes[1].set_title("Température moyenne annuelle (°C)"); axes[1].set_xlabel("Année"); axes[1].set_ylim(annuel.min().min() - 1, annuel.max().max() + 0.5)
plt.tight_layout(); plt.show()
display(annuel.round(2))

## 3. Jours présentant des conditions extrêmes (Partie 1, question 4)

In [ ]:
def top(df, col, n=3, ascending=False):
    return df.sort_values(col, ascending=ascending).groupby("city_id").head(n)[["city_id", "day_utc", col]]

blocs = {
    "Jours les plus chauds (température max, °C)": top(jour, "temperature_max"),
    "Jours les plus froids (température min, °C)": top(jour, "temperature_min", ascending=True),
    "Vent horaire maximal (m/s)": top(jour, "wind_speed_max"),
    "Journées les plus pluvieuses (mm)": top(jour.dropna(subset=["precipitation_observed"]), "precipitation_observed"),
}
for titre, d in blocs.items():
    d = d.assign(ville=d["city_id"].map(NOMS), jour=d["day_utc"].dt.date).drop(columns=["city_id", "day_utc"])
    print(titre); display(d.sort_values("ville").reset_index(drop=True).round(1))

## 4. Météo et visibilité (Partie 1, question 2)

La visibilité est une estimation de modèle. La corrélation est calculée heure par heure, avec le nombre d'heures ; Lille n'a de visibilité que depuis le 26 juin 2024, ce que le tableau montre.

In [ ]:
h = hist.dropna(subset=["weather_visibility_m"]).copy()
h["pluie"] = h["weather_precipitation_mm"] > 0
lignes = []
for ville, g in h.groupby("city_id"):
    for col, nom in (("weather_precipitation_mm", "Pluie"), ("weather_wind_speed_ms", "Vent"), ("weather_temperature_c", "Température"), ("weather_pressure_hpa", "Pression")):
        s = g[["weather_visibility_m", col]].dropna()
        if len(s) > 100 and s[col].nunique() > 1:
            rho, p = stats.spearmanr(s[col], s["weather_visibility_m"])
            lignes.append({"ville": NOMS[ville], "variable": nom, "rho": round(rho, 2), "heures": len(s), "depuis": g.hour_utc.min().date()})
display(pd.DataFrame(lignes).pivot(index="ville", columns="variable", values="rho"))
display(pd.DataFrame(lignes).groupby("ville").agg(heures=("heures", "max"), depuis=("depuis", "first")))
fig, ax = plt.subplots(figsize=(8, 4))
data = [h.loc[~h.pluie, "weather_visibility_m"] / 1000, h.loc[h.pluie, "weather_visibility_m"] / 1000]
ax.boxplot(data, tick_labels=[f"sans pluie (n={len(data[0]):,})", f"pluie (n={len(data[1]):,})"], showfliers=False)
ax.set_ylabel("Visibilité (km)"); ax.set_title("Visibilité avec et sans pluie"); plt.tight_layout(); plt.show()

## 5. Pollution sur 5 ans : tendances, dépassements des seuils OMS (Partie 3)

Concentration journalière moyenne par ville et polluant, comparée aux lignes directrices de l'OMS (24 h).

In [ ]:
poll["jour"] = poll["hour_utc"].dt.floor("D")
quot = poll.groupby(["city_id", "pollutant", "jour"], as_index=False)["value_mean"].mean()
quot["annee"] = quot["jour"].dt.year
quot["depasse"] = quot.apply(lambda r: r["value_mean"] > SEUILS_OMS.get(r["pollutant"], np.inf), axis=1)
tab = quot.groupby(["pollutant", "annee"]).agg(moyenne=("value_mean", "mean"), jours=("jour", "nunique"), part_depassement=("depasse", "mean")).reset_index()
tab["part_depassement"] = (100 * tab["part_depassement"]).round(1)
print("Part des jours au-dessus de la ligne directrice OMS (24 h), toutes villes :")
display(tab.pivot(index="annee", columns="pollutant", values="part_depassement"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for pol in ("no2", "pm25", "pm10", "o3"):
    s = mensuel[mensuel.pollutant == pol].groupby("month")["value_mean"].mean()
    axes[0].plot(s.index, s.values, label=pol.upper(), marker="o", ms=3)
axes[0].set_title("Concentration moyenne mensuelle, villes confondues (µg/m³)"); axes[0].legend(); axes[0].grid(True, linestyle="--", alpha=0.4)
cyc = quot.assign(mois=quot.jour.dt.month).groupby(["pollutant", "mois"])["value_mean"].mean().unstack(0)
for pol in ("no2", "pm25", "pm10", "o3"):
    if pol in cyc: axes[1].plot(cyc.index, cyc[pol], label=pol.upper(), marker="o", ms=3)
axes[1].set_title("Cycle saisonnier moyen (µg/m³)"); axes[1].set_xticks(range(1, 13)); axes[1].set_xlabel("Mois"); axes[1].legend(); axes[1].grid(True, linestyle="--", alpha=0.4)
plt.tight_layout(); plt.show()

## 6. Variation journalière et polluants dominants (Partie 3, questions 2 et 3)

Le polluant dominant se mesure par le **ratio à la ligne directrice OMS** : comparer des µg/m³ de gaz et de particules n'a pas de sens.

In [ ]:
poll["heure_locale"] = poll["hour_utc"].dt.tz_convert(FUSEAU).dt.hour
cycle = poll.groupby(["pollutant", "heure_locale"])["value_mean"].mean().unstack(0)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for pol in ("no2", "pm10", "pm25", "o3"):
    if pol in cycle: axes[0].plot(cycle.index, cycle[pol], marker="o", ms=3, label=pol.upper())
axes[0].set_title("Profil horaire moyen (heure locale, 2022-2026)"); axes[0].set_xticks(range(0, 24, 2)); axes[0].set_xlabel("Heure"); axes[0].set_ylabel("µg/m³"); axes[0].legend(); axes[0].grid(True, linestyle="--", alpha=0.4)
dom = quot.groupby(["city_id", "pollutant"])["value_mean"].mean().reset_index()
dom["ratio_oms"] = dom.apply(lambda r: r["value_mean"] / SEUILS_OMS.get(r["pollutant"], np.nan), axis=1)
piv = dom.pivot(index="city_id", columns="pollutant", values="ratio_oms").rename(index=NOMS)
piv.plot(kind="bar", ax=axes[1]); axes[1].axhline(1, color="red", linestyle="--", linewidth=1); axes[1].set_title("Ratio concentration moyenne / seuil OMS (1 = seuil)"); axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
print("Polluant le plus proche ou au-dessus du seuil OMS, par ville :", piv.idxmax(axis=1).to_dict())

## 7. Épisodes de pollution anormale (Partie 3, question 4)

Un jour est anormal si sa concentration journalière dépasse le 95e percentile de sa ville et de son polluant sur 2022 à 2026 ; un épisode est une suite d'au moins 2 jours anormaux consécutifs.

In [ ]:
quot = quot.sort_values(["city_id", "pollutant", "jour"]).copy()
quot["p95"] = quot.groupby(["city_id", "pollutant"])["value_mean"].transform(lambda s: s.quantile(0.95))
quot["anormal"] = quot["value_mean"] > quot["p95"]
lignes = []
for (ville, pol), g in quot.groupby(["city_id", "pollutant"]):
    run = (g["anormal"] != g["anormal"].shift()).cumsum()
    for _, bloc in g[g["anormal"]].groupby(run[g["anormal"]]):
        if len(bloc) >= 2 and pol in ("pm25", "pm10", "no2", "o3"):
            lignes.append({"ville": NOMS[ville], "polluant": pol.upper(), "debut": bloc.jour.min().date(), "jours": len(bloc), "pic_ug_m3": round(bloc.value_mean.max(), 1)})
epi = pd.DataFrame(lignes)
print(f"{len(epi)} épisodes d'au moins 2 jours au-dessus du P95")
display(epi.sort_values(["jours", "pic_ug_m3"], ascending=False).head(12).reset_index(drop=True))

## 8. Météo et pollution sur 5 ans (Partie 4, question 1)

In [ ]:
both = hist[hist["statut_jointure"] == "pollution_et_meteo"].copy()
print(f"Heures avec météo et pollution : {len(both):,} (du {both.hour_utc.min():%d/%m/%Y} au {both.hour_utc.max():%d/%m/%Y})")
METEO = {"weather_wind_speed_ms": "Vent", "weather_precipitation_mm": "Pluie", "weather_temperature_c": "Température", "weather_pressure_hpa": "Pression"}
lignes = []
for pol in ("no2", "pm25", "pm10", "o3"):
    c = f"pollution_{pol}_ugm3"
    if c not in both: continue
    for m, nom in METEO.items():
        s = both[[c, m]].dropna()
        if len(s) > 100 and s[m].nunique() > 1:
            rho, p = stats.spearmanr(s[m], s[c]); lignes.append({"polluant": pol.upper(), "météo": nom, "rho": round(rho, 2), "heures": len(s)})
cr = pd.DataFrame(lignes)
display(cr.pivot(index="polluant", columns="météo", values="rho"))

both["saison"] = both["hour_utc"].dt.month.map(lambda m: "hiver" if m in (12, 1, 2) else "printemps" if m in (3, 4, 5) else "été" if m in (6, 7, 8) else "automne")
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
bins = [0, 2, 4, 6, 8, 12, 40]; labels = ["0-2", "2-4", "4-6", "6-8", "8-12", ">12"]
for pol, color in (("no2", "#d9534f"), ("pm25", "#f0ad4e")):
    c = f"pollution_{pol}_ugm3"
    s = both[["weather_wind_speed_ms", c]].dropna(); s = s.assign(cl=pd.cut(s["weather_wind_speed_ms"], bins, labels=labels))
    g = s.groupby("cl", observed=True)[c].agg(["mean", "sem", "count"])
    axes[0].errorbar(range(len(g)), g["mean"], yerr=g["sem"], marker="o", capsize=3, label=pol.upper(), color=color)
    axes[0].set_xticks(range(len(g))); axes[0].set_xticklabels(g.index)
axes[0].set_title("Dispersion par le vent (5 ans)"); axes[0].set_xlabel("Vitesse du vent (m/s)"); axes[0].set_ylabel("µg/m³"); axes[0].legend()
sub = both[["weather_precipitation_mm", "pollution_pm10_ugm3"]].dropna()
axes[1].boxplot([sub.loc[sub.weather_precipitation_mm <= 0, "pollution_pm10_ugm3"], sub.loc[sub.weather_precipitation_mm > 0, "pollution_pm10_ugm3"]],
                tick_labels=[f"sans pluie\n(n={(sub.weather_precipitation_mm <= 0).sum():,})", f"pluie\n(n={(sub.weather_precipitation_mm > 0).sum():,})"], showfliers=False)
axes[1].set_title("PM10 avec et sans pluie"); axes[1].set_ylabel("µg/m³")
sa = both.groupby("saison")[["weather_wind_speed_ms", "pollution_no2_ugm3"]].apply(lambda g: g.dropna().corr(method="spearman").iloc[0, 1]).reindex(["hiver", "printemps", "été", "automne"])
axes[2].bar(sa.index, sa.values, color="#5bc0de"); axes[2].set_title("Corrélation vent et NO2 selon la saison"); axes[2].set_ylabel("rho de Spearman"); axes[2].axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

print("Lecture :")
for pol in ("no2", "pm25"):
    r = cr[(cr["polluant"] == pol.upper()) & (cr["météo"] == "Vent")]
    if len(r): print(f"- {pol.upper()} et vent : rho = {r.iloc[0].rho:+.2f} sur {int(r.iloc[0].heures):,} heures.")
for pol in ("pm10", "pm25"):
    c = f"pollution_{pol}_ugm3"; s = both[[c, "weather_precipitation_mm"]].dropna()
    a, b = s.loc[s.weather_precipitation_mm > 0, c], s.loc[s.weather_precipitation_mm <= 0, c]
    if len(a) > 30 and len(b) > 30:
        _, p = stats.mannwhitneyu(a, b, alternative="two-sided"); print(f"- {pol.upper()} : médiane {a.median():.1f} avec pluie ({len(a):,} h) contre {b.median():.1f} sans ({len(b):,} h), p = {p:.2g}.")

## 9. Synthèse et suites

- Les conclusions ci-dessus reposent sur 5 ans de météo modélisée et sur les stations de pollution retenues ; elles se comparent à celles du notebook 03, calculées sur deux semaines seulement.
- Le volet **mobilité** (Partie 2 et croisement vélos) attend l'archive CityBikes : il s'ajoutera à ce notebook avec la même jointure `(city_id, hour_utc)`.
- Les tables publiées dans le Data Lake (`kpis/pollution_monthly.parquet`, `kpis/history/`) alimentent l'onglet Historique du dashboard.